# SpaceX Falcon 9 Interactive Visual Analytics with Folium


## Objective
The objective of this notebook is to perform interactive geographic analysis of
SpaceX Falcon 9 launch sites using **Folium**.

The analysis will:

1. Display Falcon 9 launch sites on an interactive map.
2. Visualize successful and unsuccessful first-stage landing outcomes.
3. Examine the geographic proximity of launch sites to nearby features such as
   coastlines, cities, railways, and highways.
4. Generate interactive map outputs for the final capstone presentation.
   

## Import Required Libraries

The following libraries are used for data manipulation, geographic calculations,
and interactive map visualization.

In [8]:
!pip install folium


   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]



In [9]:
import pandas as pd
import numpy as np
import folium

from folium.plugins import MarkerCluster
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

print("Libraries imported successfully.")

Libraries imported successfully.


## Load the Falcon 9 Dataset

The wrangled Falcon 9 dataset generated in the previous stage is used as the
primary data source for the geographic analysis.

In [10]:
df = pd.read_csv("dataset_part_2.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (90, 18)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


In [11]:
print("Columns:")
print(df.columns.tolist())

Columns:
['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude', 'Class']


## Data Verification

In [12]:
required_columns = ["LaunchSite", "Latitude", "Longitude", "Class"]

for col in required_columns:
    print(f"{col}: {'FOUND' if col in df.columns else 'MISSING'}")

LaunchSite: FOUND
Latitude: FOUND
Longitude: FOUND
Class: FOUND


## Task 1 – Map Falcon 9 Launch Sites

In this task, the unique Falcon 9 launch sites are identified and displayed
on an interactive Folium map using their latitude and longitude coordinates.

In [13]:
launch_sites_df = (
    df[["LaunchSite", "Latitude", "Longitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

launch_sites_df

,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
1,VAFB SLC 4E,34.632093,-120.610829
2,KSC LC 39A,28.608058,-80.603956


### Create the Base Map

The geographic center of the launch sites is calculated and used to initialize
the Folium map.

In [22]:
map_center = [
    launch_sites_df["Latitude"].mean(),
    launch_sites_df["Longitude"].mean()
]

site_map = folium.Map(
    location=map_center,
    zoom_start=5,
    tiles=None
)


### Create the Base Interactive Map

A Folium map is initialized using the average latitude and longitude of all
unique Falcon 9 launch sites. The map will serve as the base layer for adding
launch-site markers and labels.

In [23]:
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map",
    overlay=False,
    control=True
).add_to(site_map)


site_map

### Add Launch-Site Markers

Each unique Falcon 9 launch site is added to the interactive map using a
circle marker and a text label. These markers allow the geographic locations
of the launch facilities to be identified visually.

In [24]:
for _, site in launch_sites_df.iterrows():

    # Add a circle marker for the launch site
    folium.Circle(
        location=[site["Latitude"], site["Longitude"]],
        radius=1000,
        color="#0B5ED7",
        fill=True,
        fill_color="#0B5ED7",
        fill_opacity=0.35,
        popup=site["LaunchSite"]
    ).add_to(site_map)

    # Add a text marker showing the launch-site name
    folium.Marker(
        location=[site["Latitude"], site["Longitude"]],
        icon=DivIcon(
            icon_size=(200, 36),
            icon_anchor=(0, 0),
            html=f'''
                <div style="
                    font-size: 12px;
                    font-weight: bold;
                    color: #0B5ED7;
                    white-space: nowrap;">
                    {site["LaunchSite"]}
                </div>
            '''
        )
    ).add_to(site_map)

site_map

### Verify Launch-Site Locations

The interactive map is reviewed to confirm that all unique Falcon 9 launch
sites are displayed at their corresponding geographic coordinates.

In [25]:
print(f"Number of unique launch sites mapped: {len(launch_sites_df)}")

display(
    launch_sites_df[
        ["LaunchSite", "Latitude", "Longitude"]
    ]
)

Number of unique launch sites mapped: 3


,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
1,VAFB SLC 4E,34.632093,-120.610829
2,KSC LC 39A,28.608058,-80.603956


### Save the Interactive Launch-Site Map

The completed launch-site map is saved as an HTML file so that the interactive
features can be viewed outside the Jupyter Notebook and retained as part of
the capstone project outputs.

In [26]:
site_map.save("folium_01_launch_sites.html")

print("Saved: folium_01_launch_sites.html")

Saved: folium_01_launch_sites.html


### Task 1 Observation

The interactive map shows the geographic distribution of Falcon 9 launch
facilities in the United States. Mapping the launch sites provides the spatial
foundation for examining individual launch outcomes and the proximity of each
site to surrounding geographic and transportation features.

## Task 2 – Map Successful and Unsuccessful Landing Outcomes

In this task, individual Falcon 9 launch records are displayed on the Folium map
according to their landing outcome.

- `Class = 1` represents a successful first-stage landing.
- `Class = 0` represents an unsuccessful first-stage landing.

Different marker colors are used to distinguish the two outcomes visually.

### Prepare Landing Outcome Labels

A descriptive landing-outcome label is created from the binary `Class` variable
so that each launch can be identified clearly on the interactive map.

In [27]:
launch_df = df.copy()

launch_df["LandingOutcome"] = launch_df["Class"].map({
    1: "Successful Landing",
    0: "Unsuccessful Landing"
})

launch_df[
    ["LaunchSite", "Latitude", "Longitude", "Class", "LandingOutcome"]
].head()

,LaunchSite,Latitude,Longitude,Class,LandingOutcome
0,CCAFS SLC 40,28.561857,-80.577366,0,Unsuccessful Landing
1,CCAFS SLC 40,28.561857,-80.577366,0,Unsuccessful Landing
2,CCAFS SLC 40,28.561857,-80.577366,0,Unsuccessful Landing
3,VAFB SLC 4E,34.632093,-120.610829,0,Unsuccessful Landing
4,CCAFS SLC 40,28.561857,-80.577366,0,Unsuccessful Landing


### Create the Launch-Outcome Map

A new Folium map is initialized using the same geographic center as Task 1.
Individual launch records will be added to the map using clustered markers.

In [28]:
outcome_map = folium.Map(
    location=map_center,
    zoom_start=5,
    tiles= None
)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map",
    overlay=False,
    control=True
).add_to(outcome_map)


marker_cluster = MarkerCluster().add_to(outcome_map)

### Add Launch Outcome Markers

Each launch is represented by an interactive marker:

- Green markers indicate successful landings.
- Red markers indicate unsuccessful landings.

Marker clustering is used to reduce visual overlap when several launches occurred
at the same launch site.

In [29]:
for _, row in launch_df.iterrows():

    marker_color = "green" if row["Class"] == 1 else "red"

    popup_text = (
        f"<b>Launch Site:</b> {row['LaunchSite']}<br>"
        f"<b>Landing Outcome:</b> {row['LandingOutcome']}"
    )

    folium.Marker(
        location=[row["Latitude"], row["Longitude"]],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(
            color=marker_color,
            icon="info-sign"
        )
    ).add_to(marker_cluster)

outcome_map

### Add Landing Outcome Legend

A map legend is added to make the marker colors easier to interpret.

In [30]:
legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 190px;
    background-color: white;
    border: 2px solid grey;
    z-index: 9999;
    font-size: 14px;
    padding: 10px;
    border-radius: 5px;
">
    <b>Landing Outcome</b><br>
    <span style="color:green;">●</span> Successful Landing<br>
    <span style="color:red;">●</span> Unsuccessful Landing
</div>
"""

outcome_map.get_root().html.add_child(
    folium.Element(legend_html)
)

outcome_map

### Summarize Landing Outcomes

The number of successful and unsuccessful landing outcomes is calculated to
support interpretation of the geographic visualization.

In [31]:
outcome_summary = (
    launch_df["LandingOutcome"]
    .value_counts()
    .rename_axis("Landing Outcome")
    .reset_index(name="Count")
)

outcome_summary

,Landing Outcome,Count
0,Successful Landing,60
1,Unsuccessful Landing,30


In [32]:
success_rate = launch_df["Class"].mean() * 100

print(f"Overall landing success rate: {success_rate:.2f}%")

Overall landing success rate: 66.67%


### Save the Interactive Landing-Outcome Map

The completed map is exported as an HTML file so that the interactive launch
markers and landing outcomes can be reviewed outside the notebook.

In [33]:
outcome_map.save("folium_02_launch_outcomes.html")

print("Saved: folium_02_launch_outcomes.html")

Saved: folium_02_launch_outcomes.html


### Task 2 Observation

The interactive map distinguishes successful and unsuccessful first-stage
landings at each Falcon 9 launch site. Marker clustering makes it possible to
explore multiple launches at the same location while preserving individual
mission-level landing outcomes.

This visualization provides a geographic view of landing performance and
supports comparison of launch outcomes across different launch facilities.

## Task 3 – Analyze Proximity to Nearby Geographic Features

In this task, the geographic surroundings of a Falcon 9 launch site are examined
to understand its proximity to important features such as coastlines, railways,
highways, and nearby cities.

Interactive coordinate tools are first added to the Folium map. Distances between
the selected launch site and nearby geographic features are then calculated and
visualized using markers and connecting lines.

### Add an Interactive Coordinate Tool

A mouse-position tool is added to the map so that latitude and longitude
coordinates can be inspected directly. These coordinates will be used to identify
nearby geographic features without hard-coding unverified locations.

In [34]:
from folium.plugins import MousePosition

proximity_map = folium.Map(
    location=map_center,
    zoom_start=5,
    tiles=None
)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles © Esri",
    name="Esri World Street Map",
    overlay=False,
    control=True
).add_to(proximity_map)

proximity_map

### Select a Launch Site for Proximity Analysis

One Falcon 9 launch site is selected as the reference location. Its latitude and
longitude are retrieved directly from the dataset to ensure consistency with the
previous analysis.

In [35]:
launch_sites_df

,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
1,VAFB SLC 4E,34.632093,-120.610829
2,KSC LC 39A,28.608058,-80.603956


In [36]:
selected_site_name = "KSC LC 39A"

selected_site = launch_sites_df[
    launch_sites_df["LaunchSite"] == selected_site_name
].iloc[0]

launch_site_coord = (
    selected_site["Latitude"],
    selected_site["Longitude"]
)

print("Selected launch site:", selected_site_name)
print("Coordinates:", launch_site_coord)

Selected launch site: KSC LC 39A
Coordinates: (np.float64(28.6080585), np.float64(-80.6039558))


In [37]:
folium.Marker(
    location=launch_site_coord,
    popup=selected_site_name,
    tooltip="Selected Launch Site",
    icon=folium.Icon(color="red", icon="info-sign")
).add_to(proximity_map)

proximity_map

### Calculate Geographic Distance

The Haversine formula is used to calculate the great-circle distance between
the launch site and a nearby geographic feature using their latitude and
longitude coordinates.

In [38]:
from math import radians, sin, cos, sqrt, atan2

def calculate_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two geographic coordinates.
    Distance is returned in kilometers.
    """

    earth_radius_km = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2)**2
        + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return earth_radius_km * c

### Identify Nearby Geographic Features

Using the interactive coordinate display, identify representative nearby points
for the following geographic features:

- Coastline
- Railway
- Highway
- Nearby city

The coordinates should be obtained directly from the interactive map and then
entered into the next code cell.

In [39]:
nearby_features = {
    "Atlantic Coastline": (28.622399, -80.607393),
    "NASA Railroad": (28.644885, -80.703083),
    "SR 3 / Kennedy Parkway": (28.644888, -80.700208),
    "Titusville": (28.609330, -80.807600)
}

In [40]:
launch_site_coord = (28.608402, -80.604201)

### Calculate Distances to Nearby Features

The distance from the selected launch site to each identified geographic feature
is calculated in kilometers.

In [41]:
distance_results = []

for feature_name, feature_coord in nearby_features.items():

    distance_km = calculate_distance(
        launch_site_coord[0],
        launch_site_coord[1],
        feature_coord[0],
        feature_coord[1]
    )

    distance_results.append({
        "Feature": feature_name,
        "Distance (km)": round(distance_km, 2)
    })

distance_df = pd.DataFrame(distance_results)

distance_df

,Feature,Distance (km)
0,Atlantic Coastline,1.59
1,NASA Railroad,10.47
2,SR 3 / Kennedy Parkway,10.21
3,Titusville,19.86


### Visualize Distances on the Interactive Map

Markers are added for each nearby geographic feature. A line connects each
feature to the selected launch site, and the calculated distance is displayed
on the map.

In [42]:
for feature_name, feature_coord in nearby_features.items():

    distance_km = calculate_distance(
        launch_site_coord[0],
        launch_site_coord[1],
        feature_coord[0],
        feature_coord[1]
    )

    folium.Marker(
        location=feature_coord,
        popup=f"{feature_name}: {distance_km:.2f} km",
        tooltip=feature_name
    ).add_to(proximity_map)

    folium.PolyLine(
        locations=[
            launch_site_coord,
            feature_coord
        ],
        weight=2,
        opacity=0.8,
        tooltip=f"{feature_name}: {distance_km:.2f} km"
    ).add_to(proximity_map)

proximity_map

### Save the Proximity Analysis Map

The completed proximity-analysis map is exported as an HTML file so that the
interactive markers, distance lines, and geographic relationships can be
reviewed outside the notebook.

In [43]:
proximity_map.save("folium_03_proximity_analysis.html")

print("Saved: folium_03_proximity_analysis.html")

Saved: folium_03_proximity_analysis.html


### Proximity Analysis Results

The calculated distances between the selected launch site and nearby geographic
features are summarized below.

In [44]:
distance_df

,Feature,Distance (km)
0,Atlantic Coastline,1.59
1,NASA Railroad,10.47
2,SR 3 / Kennedy Parkway,10.21
3,Titusville,19.86


### Task 3 Observation

The proximity analysis shows that the selected Falcon 9 launch site is located
near the Atlantic coastline and is also accessible to transportation
infrastructure and nearby urban areas.

The distance lines and interactive markers provide a spatial view of how the
launch facility is positioned relative to surrounding geographic features.
These relationships are useful for understanding the operational setting of
the launch site.

## Slide-Ready Methodology Statement

**Interactive geographic analysis was performed using Folium.** Falcon 9 launch sites were mapped using latitude and longitude coordinates, while clustered mission-level markers were used to distinguish successful and unsuccessful landing outcomes. Interactive coordinate tools, geographic markers, and distance lines were then used to examine the proximity of a selected launch site to nearby coastlines, transportation infrastructure, and urban areas.

## Saved Interactive Map Outputs

1. `folium_01_launch_sites.html` – Falcon 9 launch-site locations
2. `folium_02_launch_outcomes.html` – Successful and unsuccessful landing outcomes
3. `folium_03_proximity_analysis.html` – Geographic proximity analysis


In [45]:
import os

output_files = [
    "folium_01_launch_sites.html",
    "folium_02_launch_outcomes.html",
    "folium_03_proximity_analysis.html"
]

for file in output_files:
    print(
        f"{file}:",
        "SAVED" if os.path.exists(file) else "MISSING"
    )

folium_01_launch_sites.html: SAVED
folium_02_launch_outcomes.html: SAVED
folium_03_proximity_analysis.html: SAVED
